# Data construction — corporate default panel
 
Builds the firm-year analysis panel from three WRDS extracts: Compustat annual fundamentals (via the CRSP/Compustat Merged link), the CRSP monthly stock file, and the CRSP delisting file.
 
**Design choices that matter for credit-risk validation**
 
- **Point-in-time.** The default indicator is placed on a firm's *last complete fiscal year* and kept only when that year precedes the delisting year by **one or two years**, so every predictor is observed strictly before the event.
- **No look-ahead in preprocessing.** Winsorization bounds are computed cross-sectionally *within each fiscal year*, so future data never influences current-year clipping.
 
**Output:** `data/intermediate/panel_clean.csv` — one row per firm-fiscal year, consumed by the dynamic-logit and XGBoost notebooks.
 
> Raw WRDS files are licensed and **not** included in this repository. See `data/README.md` for the download specifications.

In [1]:
import os
from pathlib import Path
 
import numpy as np
import pandas as pd
from pandas.tseries.offsets import MonthEnd

## Configuration
 
Set `DATA_DIR` to the folder that contains `raw/` and `intermediate/`.

In [2]:
# DATA_DIR must contain raw/ and intermediate/.
# Default assumes the repo layout (notebooks/ and data/ are siblings), so running
# from notebooks/ finds ../data. Override without editing this file:
#     export DATA_DIR=/absolute/path/to/your/Credit_risk/Data
DATA_DIR = Path(os.environ.get("DATA_DIR", "../data"))
RAW = DATA_DIR / "raw"
INTERIM = DATA_DIR / "intermediate"
INTERIM.mkdir(parents=True, exist_ok=True)
 
# Fail early and clearly if DATA_DIR is wrong. Raising rather than printing keeps
# local absolute paths out of committed notebook output.
if not RAW.exists():
    raise FileNotFoundError(
        "No raw/ folder found under DATA_DIR. Set DATA_DIR to the folder "
        "containing raw/ and intermediate/, e.g. export DATA_DIR=/path/to/Data"
    )

## 1. Compustat annual fundamentals

Load the annual file, exclude financials (SIC 6000–6999) and regulated utilities (SIC 4900–4999), and construct the fiscal year-end date from `fyear` and `fyr` (firms have different fiscal year-ends).

In [3]:
annual = pd.read_csv(RAW / "annual.csv")
annual.columns = annual.columns.str.lower()
 
# Exclude financials and regulated utilities (different balance-sheet structures)
annual = annual[~annual["sich"].between(6000, 6999)]
annual = annual[~annual["sich"].between(4900, 4999)]
 
# Sort within firm (needed for the sales-growth lag); require fiscal year and month
annual = annual.sort_values(["gvkey", "fyear"]).reset_index(drop=True)
annual = annual.dropna(subset=["fyr", "fyear"])
 
# Fiscal year-end date = end of (fyear, fyr) month
annual["datadate"] = (
    pd.to_datetime(
        annual["fyear"].astype(int).astype(str) + "-" + annual["fyr"].astype(int).astype(str),
        format="%Y-%m",
    )
    + MonthEnd(0)
)
print(f"Loaded {len(annual):,} firm-year observations after industry exclusions")
print(f"Fiscal years {int(annual['fyear'].min())}-{int(annual['fyear'].max())}")

Loaded 228,814 firm-year observations after industry exclusions
Fiscal years 1980-2022


## 2. Accounting (book-based) features

In [4]:
# Book leverage: (long-term + short-term debt) / total assets
annual["leverage"] = ((annual["dltt"] + annual["dlc"]) / annual["at"]).replace([np.inf, -np.inf], np.nan)

# Interest coverage: EBITDA / interest expense.
# A firm with no interest expense has no debt-servicing burden -> maximal coverage,
# so we assign a high capped value (99) rather than 0, which would wrongly mark it as distressed.
annual["int_coverage"] = np.where(annual["xint"] > 0, annual["oibdp"] / annual["xint"], 99)

annual["roa"] = annual["ni"] / annual["at"]
annual["current_ratio"] = (annual["act"] / annual["lct"]).replace([np.inf, -np.inf], np.nan)
annual["cash_ratio"] = annual["che"] / annual["at"]
annual["log_at"] = np.log(annual["at"].where(annual["at"] > 0))

# Sales growth (YoY, within firm). Kept for the completeness filter / robustness;
# NOT one of the 13 final model predictors.
annual["sale_lag"] = annual.groupby("gvkey")["sale"].shift(1)
annual["sales_growth"] = (
    (annual["sale"] - annual["sale_lag"]) / annual["sale_lag"].abs()
).replace([np.inf, -np.inf], np.nan)

In [5]:
def winsorize(df, cols, lower=0.01, upper=0.99):
    """Clip each column to within-year percentile bounds (cross-sectional, by fiscal year).

    Computing bounds per fiscal year avoids look-ahead: future years never affect
    current-year clipping.
    """
    for col in cols:
        df[col] = df.groupby("fyear")[col].transform(
            lambda x: x.clip(lower=x.quantile(lower), upper=x.quantile(upper))
        )
    return df


acct_feats = ["leverage", "int_coverage", "roa", "current_ratio", "cash_ratio", "log_at", "sales_growth"]
annual = winsorize(annual, acct_feats)

# Structural "has interest-bearing debt" flag (diagnostic only; not a model feature)
annual["has_debt"] = (annual["xint"].fillna(0) > 0).astype(int)
print("Accounting features constructed and winsorized (annual, 1st/99th pct)")

Accounting features constructed and winsorized (annual, 1st/99th pct)


## 3. CRSP monthly stock data

Market capitalization, the trailing 12-month return, and annualized equity volatility, all from monthly returns (no daily file required).

In [6]:
stock = pd.read_csv(RAW / "stock.csv")
stock.columns = stock.columns.str.lower()
stock["mthcaldt"] = pd.to_datetime(stock["mthcaldt"])

# Market cap ($MM): |price| x shares outstanding (shrout in thousands)
stock["mktcap"] = stock["mthprc"].abs() * stock["shrout"] / 1000

# Chronological order per security for the rolling windows
stock = stock.sort_values(["permno", "mthcaldt"]).reset_index(drop=True)

stock["trail_12m_ret"] = stock.groupby("permno")["mthret"].transform(
    lambda x: (1 + x).rolling(12).apply(np.prod, raw=True) - 1
)
stock["eq_vol"] = stock.groupby("permno")["mthret"].transform(
    lambda x: x.rolling(12).std() * np.sqrt(12)
)
print(f"Loaded {len(stock):,} monthly stock observations")

Loaded 3,806,920 monthly stock observations


## 4. Merge accounting and market data at fiscal year-end

Each firm-year is matched to its CRSP record in the fiscal year-end month, which aligns non-December fiscal years with the correct market data.

In [7]:
annual["merge_ym"] = annual["datadate"].dt.to_period("M")
stock["merge_ym"] = stock["mthcaldt"].dt.to_period("M")

merged = annual.merge(
    stock[["permno", "merge_ym", "mktcap", "trail_12m_ret", "eq_vol", "mthprc"]],
    left_on=["lpermno", "merge_ym"],
    right_on=["permno", "merge_ym"],
    how="left",
)
print(
    f"Merged sample: {len(merged):,} rows "
    f"({(1 - merged['mktcap'].isna().mean()) * 100:.1f}% with stock data)"
)

Merged sample: 231,576 rows (97.9% with stock data)


## 5. Market-valued (Campbell–Hilscher–Szilagyi) features

Profitability, leverage, and liquidity scaled by market-valued assets (market cap + total liabilities), plus market-to-book and the capped log price.

In [8]:
mva = merged["mktcap"] + merged["lt"]  # market-valued assets
merged["nimta"] = (merged["ni"] / mva).replace([np.inf, -np.inf], np.nan)
merged["tlmta"] = (merged["lt"] / mva).replace([np.inf, -np.inf], np.nan)
merged["cashmta"] = (merged["che"] / mva).replace([np.inf, -np.inf], np.nan)

merged["book_equity"] = merged["at"] - merged["lt"]
merged["mtb"] = (merged["mktcap"] / merged["book_equity"]).replace([np.inf, -np.inf], np.nan)

# Market leverage: debt / (debt + market cap). Kept for robustness; not a final predictor.
debt = merged["dltt"] + merged["dlc"]
merged["mkt_leverage"] = (debt / (debt + merged["mktcap"])).replace([np.inf, -np.inf], np.nan)

# Log stock price, capped at log(15): penny-stock prices are noisy (CHS treatment of low-priced stocks)
merged["price"] = np.log(merged["mthprc"].abs() + 1).clip(upper=np.log(15))

market_feats = ["mktcap", "trail_12m_ret", "eq_vol", "mtb", "mkt_leverage", "nimta", "tlmta", "cashmta", "price"]
merged = winsorize(merged, market_feats)
print("Market-valued features constructed and winsorized")

Market-valued features constructed and winsorized


## 6. Default events (point-in-time)

Default is a CRSP distress delisting (`BKPY`, `INSC`, `DELQ`). The indicator is placed on a firm's **last** observed fiscal year and retained only when that year precedes the delisting year by **one or two years**, so all predictors are observed strictly before the event. This is the key point-in-time guard.

In [9]:
delisting = pd.read_csv(RAW / "delisting.csv")
delisting.columns = delisting.columns.str.lower()
delisting["delistingdt"] = pd.to_datetime(delisting["delistingdt"])
delisting["delyear"] = delisting["delistingdt"].dt.year

# Distress default: bankruptcy, insolvency, delinquency
delisting["default"] = delisting["delreasontype"].isin(["BKPY", "INSC", "DELQ"]).astype(int)
print(f"Distress delistings (BKPY+INSC+DELQ): {int(delisting['default'].sum()):,}")

Distress delistings (BKPY+INSC+DELQ): 3,315


In [10]:
delist_clean = delisting[["permno", "delyear", "default"]]
merged = merged.merge(delist_clean, left_on="lpermno", right_on="permno", how="left")
merged["default"] = merged["default"].fillna(0)
merged["delyear"] = merged["delyear"].fillna(0).astype(int)

# Place the event on the firm's last observed fiscal year
last_obs = merged.groupby("lpermno")["fyear"].max().rename("last_fyear")
merged = merged.merge(last_obs, on="lpermno", how="left")
merged["default_flag"] = (
    (merged["default"] == 1) & (merged["fyear"] == merged["last_fyear"])
).astype(int)

# POINT-IN-TIME RESTRICTION: keep a defaulting firm only if its last fiscal year
# precedes the delisting year by 1-2 years (predictors strictly before the event).
merged["diff_year"] = merged["last_fyear"] - merged["delyear"]
bad = merged[(merged["default_flag"] == 1) & (~merged["diff_year"].isin([-1, -2]))]["lpermno"].unique()
merged = merged[~merged["lpermno"].isin(bad)].reset_index(drop=True)
merged = merged.drop(columns=["diff_year", "default"])
print(f"Dropped {len(bad):,} firms with inconsistent event timing; {len(merged):,} rows remain")

Dropped 337 firms with inconsistent event timing; 229,085 rows remain


## 7. Deduplicate and write the final panel

When one Compustat firm links to multiple CRSP securities in a fiscal year, keep the highest-market-cap (primary) share class. The panel then requires all model inputs to be non-missing.

In [11]:
# Keep the primary share class (highest market cap) per firm-year
merged = merged.sort_values(["gvkey", "fyear", "mktcap"], ascending=[True, True, False])
merged = merged.drop_duplicates(subset=["gvkey", "fyear"], keep="first")
 
# The 13 predictors used by both models
model_features = [
    "leverage", "int_coverage", "roa", "current_ratio", "cash_ratio", "log_at",
    "nimta", "tlmta", "cashmta", "price", "trail_12m_ret", "eq_vol", "mtb",
]
 
# Completeness filter. NOTE: sales_growth, mkt_leverage, and mktcap are also required
# here (carried for robustness checks) even though they are not among the 13 model
# predictors. Requiring sales_growth (a within-firm lag) drops first-year-per-firm rows.
# Dropping these from `required` would enlarge the sample and change every
# result in the paper, so the list is deliberate rather than incidental.
required = model_features + ["sales_growth", "mkt_leverage", "mktcap"]
df = merged.dropna(subset=required).reset_index(drop=True)
 
print("===== FINAL PANEL =====")
print(f"  Firm-years:     {len(df):,}")
print(f"  Unique firms:   {df['gvkey'].nunique():,}")
print(f"  Default events: {int(df['default_flag'].sum()):,}")
print(f"  Event rate:     {df['default_flag'].mean() * 100:.3f}%")
 
df.to_csv(INTERIM / "panel_clean.csv", index=False)
print("Saved intermediate/panel_clean.csv")

===== FINAL PANEL =====
  Firm-years:     157,556
  Unique firms:   16,928
  Default events: 1,870
  Event rate:     1.187%
Saved intermediate/panel_clean.csv


## 8. Table 2 — summary statistics

Mean (standard deviation) of the 13 model features on the cleaned, winsorized panel, across the out-of-time train/test windows. These are the exact numbers reported in the paper.

In [12]:
feature_labels = [
    ("leverage", "Book leverage"), ("int_coverage", "Interest coverage"), ("roa", "ROA"),
    ("current_ratio", "Current ratio"), ("cash_ratio", "Cash/assets"), ("log_at", "Log assets"),
    ("nimta", "NIMTA"), ("tlmta", "TLMTA"), ("cashmta", "CASHMTA"), ("price", "Log price"),
    ("trail_12m_ret", "Trailing 12-month return"), ("eq_vol", "Equity volatility"), ("mtb", "Market-to-book"),
]

windows = {
    "Train": df["fyear"] <= 2005,
    "2006-2010": df["fyear"].between(2006, 2010),
    "2011-2015": df["fyear"].between(2011, 2015),
    "Post-2015": df["fyear"] >= 2016,
}

rows = [{"Feature": "N", **{w: f"{int(m.sum())}" for w, m in windows.items()}}]
for col, label in feature_labels:
    rows.append({"Feature": label, **{w: f"{df.loc[m, col].mean():.3f} ({df.loc[m, col].std():.3f})" for w, m in windows.items()}})

table2 = pd.DataFrame(rows).set_index("Feature")
print(table2.to_string())

                                     Train         2006-2010         2011-2015        Post-2015
Feature                                                                                        
N                                   103154             17298             15475            21629
Book leverage                0.238 (0.209)     0.203 (0.215)     0.223 (0.219)    0.287 (0.246)
Interest coverage         26.614 (118.423)  54.629 (219.947)  49.043 (226.417)  3.244 (259.204)
ROA                         -0.051 (0.288)    -0.046 (0.280)    -0.037 (0.256)   -0.095 (0.359)
Current ratio                2.784 (2.917)     2.831 (2.697)     2.871 (2.914)    2.939 (3.430)
Cash/assets                  0.160 (0.196)     0.219 (0.227)     0.212 (0.222)    0.237 (0.251)
Log assets                   4.823 (2.161)     6.206 (2.092)     6.615 (2.116)    6.833 (2.199)
NIMTA                       -0.024 (0.172)    -0.026 (0.179)    -0.018 (0.147)   -0.058 (0.224)
TLMTA                        0.397 (0.25